# Зовнішня HTTP-перевірка лабораторної 1
Запустіть fixture-сервер на 8011 і live-сервер на 8012 за README або `python scripts/verify.py`. Тут немає імпорту app, TestClient чи викликів Groq: тільки справжні HTTP-запити. Тестове сховище — пам'ять окремого процесу. Повторний прохід порівнює приріст від наявного стану, не видаляє чужі записи. Timeout 25 секунд > 15 секунд провайдера. Очікування незалежні, взяті зі специфікації 5a42f6f.


In [1]:

import os, json
from pathlib import Path
from uuid import UUID
import requests
from IPython.display import Markdown, display
BASE_URL=os.getenv('VERIFY_URL','http://127.0.0.1:8011')
LIVE_URL=os.getenv('LIVE_VERIFY_URL','http://127.0.0.1:8012')
ROOT=Path.cwd() if (Path.cwd()/'app').exists() else Path.cwd().parent
print('Контрольований сервер:',BASE_URL,'; live:',LIVE_URL)
rows=[]
def request(method,path,payload=None,base=BASE_URL):
    response=requests.request(method,base+path,json=payload,timeout=25)
    return response.status_code,response.json()
def snapshot(base=BASE_URL):
    code,data=request('GET','/videos',base=base)
    assert code==200 and isinstance(data,list)
    for record in data:
        assert set(record)=={'id','title','topic','duration_minutes'}
        UUID(record['id'])
    assert len({r['id'] for r in data})==len(data)
    return data
def run(name,ac,source,payload=None,expected=None,code_expected=201,reason=None,base=BASE_URL,path='/videos/ai'):
    row={'scenario':name,'criterion':ac,'source':source,'expected_http':code_expected,'status':'failed'}
    try:
        before=snapshot(base)
        if name=='initial_list':
            row.update(actual_http=200,before=len(before),after=len(before),status='passed',records=before)
        else:
            code,body=request('POST',path,payload,base)
            after=snapshot(base)
            row.update(actual_http=code,response=body,before=len(before),after=len(after))
            assert code==code_expected, f'HTTP {code} != {code_expected}'
            if code_expected==201:
                UUID(body['id'])
                assert {k:body[k] for k in expected}==expected
                assert body['id'] not in {r['id'] for r in before}
                assert after==before+[body], 'Список змінився не рівно на один правильний запис'
            else:
                assert after==before, 'Відмова змінила кількість або вміст записів'
                if reason: assert body['detail']['status']==reason
            row['status']='passed'
    except Exception as exc:
        row['reason']=f'{type(exc).__name__}: {exc}'
    rows.append(row)
    print(json.dumps(row,ensure_ascii=False))
def fields(title='Python',duration=15):
    return {'title':title,'topic':'цикли','duration_minutes':duration}
def text(title='Python',duration=15):
    return {'text':f'Відео {title}, тема цикли, тривалість {duration} хвилин'}


Контрольований сервер: http://127.0.0.1:8011 ; live: http://127.0.0.1:8012


In [2]:

run('initial_list','AC1','не застосовується',code_expected=200)
run('manual','AC1','не застосовується',fields(),fields(),path='/videos')
run('manual_invalid','AC1','не застосовується',fields(duration=-5),code_expected=422,path='/videos')
run('ai_valid','AC2','підготовлена відповідь',text(),fields())
run('incomplete','AC3','не застосовується',{'text':'Відео Python, тема цикли'},code_expected=422,reason='needs_clarification')
run('ambiguous','AC3','не застосовується',{'text':'Відео Python, тема цикли, тривалість 15 або 20 хвилин'},code_expected=422,reason='needs_clarification')
run('ai_null','AC3','підготовлена відповідь',text('FixtureNull'),code_expected=422,reason='needs_clarification')
run('invalid_proposal','AC4','підготовлена відповідь',text('FixtureInvalid'),code_expected=422,reason='invalid_proposal')
run('semantic_mismatch','AC4','підготовлена відповідь',text('FixtureWrong',12),code_expected=422,reason='semantic_mismatch')
run('model_down','AC5','імітований збій',text('FixtureDown'),code_expected=503,reason='model_unavailable')
run('manual_after_failure','AC5','не застосовується',fields('Після збою'),fields('Після збою'),path='/videos')


{"scenario": "initial_list", "criterion": "AC1", "source": "не застосовується", "expected_http": 200, "status": "passed", "actual_http": 200, "before": 3, "after": 3, "records": [{"title": "Python", "topic": "цикли", "duration_minutes": 15, "id": "e4edfeec-18e3-4cb7-b467-45fccfe244e0"}, {"title": "Python", "topic": "цикли", "duration_minutes": 15, "id": "c3134686-1e2f-4578-94a8-babf72f8d2d6"}, {"title": "Після збою", "topic": "цикли", "duration_minutes": 15, "id": "ceaa43fd-8354-48fb-8d4c-54ffc5aec8ec"}]}
{"scenario": "manual", "criterion": "AC1", "source": "не застосовується", "expected_http": 201, "status": "passed", "actual_http": 201, "response": {"title": "Python", "topic": "цикли", "duration_minutes": 15, "id": "7fe1add8-2f42-4ce2-ae39-a7250c9eb3ee"}, "before": 3, "after": 4}
{"scenario": "manual_invalid", "criterion": "AC1", "source": "не застосовується", "expected_http": 422, "status": "passed", "actual_http": 422, "response": {"detail": [{"type": "greater_than", "loc": ["body"

{"scenario": "ambiguous", "criterion": "AC3", "source": "не застосовується", "expected_http": 422, "status": "passed", "actual_http": 422, "response": {"detail": {"status": "needs_clarification", "reason": "Вкажіть: Відео НАЗВА, тема ТЕМА, тривалість N хвилин (1–1440)."}}, "before": 5, "after": 5}


{"scenario": "ai_null", "criterion": "AC3", "source": "підготовлена відповідь", "expected_http": 422, "status": "passed", "actual_http": 422, "response": {"detail": {"status": "needs_clarification", "reason": "Модель не визначила всі обов’язкові поля."}}, "before": 5, "after": 5}
{"scenario": "invalid_proposal", "criterion": "AC4", "source": "підготовлена відповідь", "expected_http": 422, "status": "passed", "actual_http": 422, "response": {"detail": {"status": "invalid_proposal", "reason": "Пропозиція не відповідає схемі відео."}}, "before": 5, "after": 5}
{"scenario": "semantic_mismatch", "criterion": "AC4", "source": "підготовлена відповідь", "expected_http": 422, "status": "passed", "actual_http": 422, "response": {"detail": {"status": "semantic_mismatch", "reason": "Поля моделі суперечать явно вказаним даним повідомлення."}}, "before": 5, "after": 5}
{"scenario": "model_down", "criterion": "AC5", "source": "імітований збій", "expected_http": 503, "status": "passed", "actual_http":

In [3]:

# Реальний HTTP-запит до окремого live-сервера: ключ відомий лише серверу.
run('live_ai','AC2','реальна модель',text('Python'),fields('Python'),base=LIVE_URL)
live=rows[-1]
if live.get('actual_http')==503 and live.get('before')==live.get('after'):
    live['status']='not_run'
    live['reason']='Провайдер недоступний; живу інтеграцію не підтверджено.'
elif 'actual_http' not in live:
    live['status']='not_run'
    live['reason']='Live-сервер недоступний; HTTP live-перевірка не виконана.'
print('Остаточний live-результат:',json.dumps(live,ensure_ascii=False))


{"scenario": "live_ai", "criterion": "AC2", "source": "реальна модель", "expected_http": 201, "status": "passed", "actual_http": 201, "response": {"title": "Python", "topic": "цикли", "duration_minutes": 15, "id": "594a42e6-49f1-455b-ab7b-9c10afed0e14"}, "before": 1, "after": 2}
Остаточний live-результат: {"scenario": "live_ai", "criterion": "AC2", "source": "реальна модель", "expected_http": 201, "status": "passed", "actual_http": 201, "response": {"title": "Python", "topic": "цикли", "duration_minutes": 15, "id": "594a42e6-49f1-455b-ab7b-9c10afed0e14"}, "before": 1, "after": 2}


In [4]:

table='|Критерій|Сценарій|Джерело|Очікуваний HTTP|Фактичний|До→після|Результат|\n|---|---|---|---|---|---|---|\n'
for r in rows:
    table+=f"|{r['criterion']}|{r['scenario']}|{r['source']}|{r['expected_http']}|{r.get('actual_http','—')}|{r.get('before','—')}→{r.get('after','—')}|{r['status']}|\n"
display(Markdown(table))
summary={s:sum(r['status']==s for r in rows) for s in ['passed','failed','not_run']}
print(summary)
(ROOT/'reports').mkdir(exist_ok=True)
(ROOT/'reports'/'http-verification.json').write_text(json.dumps({'base_url':BASE_URL,'live_url':LIVE_URL,'rows':rows,'summary':summary},ensure_ascii=False,indent=2),encoding='utf-8')
assert all(r['status']=='passed' for r in rows if r['scenario']!='live_ai'), 'Детерміновані сценарії не всі пройшли — дивіться таблицю'


|Критерій|Сценарій|Джерело|Очікуваний HTTP|Фактичний|До→після|Результат|
|---|---|---|---|---|---|---|
|AC1|initial_list|не застосовується|200|200|3→3|passed|
|AC1|manual|не застосовується|201|201|3→4|passed|
|AC1|manual_invalid|не застосовується|422|422|4→4|passed|
|AC2|ai_valid|підготовлена відповідь|201|201|4→5|passed|
|AC3|incomplete|не застосовується|422|422|5→5|passed|
|AC3|ambiguous|не застосовується|422|422|5→5|passed|
|AC3|ai_null|підготовлена відповідь|422|422|5→5|passed|
|AC4|invalid_proposal|підготовлена відповідь|422|422|5→5|passed|
|AC4|semantic_mismatch|підготовлена відповідь|422|422|5→5|passed|
|AC5|model_down|імітований збій|503|503|5→5|passed|
|AC5|manual_after_failure|не застосовується|201|201|5→6|passed|
|AC2|live_ai|реальна модель|201|201|1→2|passed|


{'passed': 12, 'failed': 0, 'not_run': 0}


## Висновок
Перший notebook порівнює правила та модель. Pytest перевіряє внутрішню логіку на контрольованих пропозиціях. Цей notebook перевіряє маршрути, JSON, реальне HTTP-з'єднання й зміни даних окремого процесу. Fixture-успіхи не доводять якість Groq; для цього виділено live_ai. Відмова порівнює повний список, а не тільки довжину. Неуспішні перевірки залишаються в таблиці. Ручний шлях після збою перевірено окремо.
